<a href="https://colab.research.google.com/github/JSJeong-me/MCP/blob/main/02%20Agent%20Tool/3_mcp_module3_tools_fastmcp_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 3 실습: MCP Tools 구현 with FastMCP

이 노트북은 다음 3가지를 한 번에 실습합니다.

1. **도구(Tools)의 개념 이해**
2. **입력 스키마(JSON Schema) 확인**
3. **Python(FastMCP)로 첫 번째 MCP 도구 제작 및 호출**

구성:
- `add_numbers`: 가장 작은 계산형 tool
- `save_memo`: SQLite DB에 데이터를 저장하는 실제 action tool


## 실행 순서

아래 순서대로 실행하세요.

1. **FastMCP 설치**
2. **기본 import**
3. **MCP Server와 Tools 정의**
4. **in-memory Client로 tool 목록 조회**
5. **`add_numbers` tool 호출**
6. **`save_memo` tool 호출**
7. **SQLite DB 저장 결과 확인**


In [1]:
# 1) 설치
!pip -q install -U fastmcp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.6/728.6 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.3/204.3 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.3/152.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 15.2 MB/s eta 0:00:00


In [2]:
# 2) 기본 import
from fastmcp import FastMCP, Client
import sqlite3
import json
from pathlib import Path

print('import 완료')

import 완료


## 1) MCP Server와 Tools 정의

FastMCP에서는 일반 Python 함수를 `@mcp.tool`로 감싸서 MCP tool로 노출할 수 있습니다.
함수 이름, docstring, 타입 힌트가 tool의 이름/설명/입력 스키마 생성에 사용됩니다.


In [3]:
mcp = FastMCP('Module3-Tools-Demo')

DB_PATH = Path('mcp_demo_memo.db')

def init_db():
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute(
        '''
        CREATE TABLE IF NOT EXISTS memos (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT NOT NULL,
            content TEXT NOT NULL
        )
        '''
    )
    conn.commit()
    conn.close()

init_db()

@mcp.tool
def add_numbers(a: int, b: int) -> int:
    """두 정수를 더해서 결과를 반환합니다."""
    return a + b

@mcp.tool
def save_memo(title: str, content: str) -> dict:
    """제목과 본문을 SQLite DB에 저장하고 저장 결과를 반환합니다."""
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute(
        'INSERT INTO memos (title, content) VALUES (?, ?)',
        (title, content)
    )
    conn.commit()
    row_id = cur.lastrowid
    conn.close()
    return {
        'saved': True,
        'id': row_id,
        'title': title
    }

print('Server와 Tool 정의 완료')

Server와 Tool 정의 완료


## 2) Tool 스키마 확인

FastMCP Client는 서버 인스턴스를 직접 받아 **in-memory transport**로 연결할 수 있습니다.
이 방식은 별도 프로세스나 네트워크 없이 로컬 테스트에 적합합니다.

이 셀에서는 서버가 노출하는 tool 목록과 자동 생성된 입력 스키마를 확인합니다.


In [4]:
def to_dict(obj):
    if hasattr(obj, 'model_dump'):
        return obj.model_dump()
    if isinstance(obj, dict):
        return obj
    result = {}
    for key in ['name', 'description', 'inputSchema', 'input_schema']:
        if hasattr(obj, key):
            result[key] = getattr(obj, key)
    return result

async def show_tools():
    client = Client(mcp)
    async with client:
        tools = await client.list_tools()
        print(f'도구 개수: {len(tools)}')
        for i, tool in enumerate(tools, start=1):
            print('\n' + '=' * 70)
            print(f'TOOL #{i}')
            print('=' * 70)
            print(json.dumps(to_dict(tool), ensure_ascii=False, indent=2, default=str))

await show_tools()

도구 개수: 2

TOOL #1
{
  "name": "add_numbers",
  "title": null,
  "description": "두 정수를 더해서 결과를 반환합니다.",
  "inputSchema": {
    "additionalProperties": false,
    "properties": {
      "a": {
        "type": "integer"
      },
      "b": {
        "type": "integer"
      }
    },
    "required": [
      "a",
      "b"
    ],
    "type": "object"
  },
  "outputSchema": {
    "properties": {
      "result": {
        "type": "integer"
      }
    },
    "required": [
      "result"
    ],
    "type": "object",
    "x-fastmcp-wrap-result": true
  },
  "icons": null,
  "annotations": null,
  "meta": {
    "fastmcp": {
      "tags": []
    }
  },
  "execution": null
}

TOOL #2
{
  "name": "save_memo",
  "title": null,
  "description": "제목과 본문을 SQLite DB에 저장하고 저장 결과를 반환합니다.",
  "inputSchema": {
    "additionalProperties": false,
    "properties": {
      "title": {
        "type": "string"
      },
      "content": {
        "type": "string"
      }
    },
    "required": [
      "title",
    

## 3) 첫 번째 MCP tool 호출: add_numbers

이제 클라이언트가 실제로 tool을 호출합니다.
입력은 사전(dict) 형태로 전달합니다.


In [5]:
async def run_add_demo():
    client = Client(mcp)
    async with client:
        result = await client.call_tool('add_numbers', {'a': 7, 'b': 8})
        print('raw result:', result)
        print('structured data:', getattr(result, 'data', None))
        content = getattr(result, 'content', None)
        if content:
            print('content blocks:', content)

await run_add_demo()

raw result: CallToolResult(content=[TextContent(type='text', text='15', annotations=None, meta=None)], structured_content={'result': 15}, meta={'fastmcp': {'wrap_result': True}}, data=15, is_error=False)
structured data: 15
content blocks: [TextContent(type='text', text='15', annotations=None, meta=None)]


## 4) 실제 action tool 호출: save_memo

이번에는 계산이 아니라 DB에 실제로 데이터를 저장하는 tool을 호출합니다.
즉, tool이 단순 함수가 아니라 **실제 행동(API 호출, DB 업데이트 등)** 을 수행할 수 있음을 보여줍니다.


In [6]:
async def run_save_demo():
    client = Client(mcp)
    async with client:
        result = await client.call_tool(
            'save_memo',
            {
                'title': 'MCP Tool 실습',
                'content': 'FastMCP로 첫 번째 tool을 만들고 SQLite에 저장했습니다.'
            }
        )
        print('raw result:', result)
        print('structured data:', getattr(result, 'data', None))
        content = getattr(result, 'content', None)
        if content:
            print('content blocks:', content)

await run_save_demo()

raw result: CallToolResult(content=[TextContent(type='text', text='{"saved":true,"id":1,"title":"MCP Tool 실습"}', annotations=None, meta=None)], structured_content={'saved': True, 'id': 1, 'title': 'MCP Tool 실습'}, meta=None, data={'saved': True, 'id': 1, 'title': 'MCP Tool 실습'}, is_error=False)
structured data: {'saved': True, 'id': 1, 'title': 'MCP Tool 실습'}
content blocks: [TextContent(type='text', text='{"saved":true,"id":1,"title":"MCP Tool 실습"}', annotations=None, meta=None)]


## 5) DB 저장 결과 직접 확인

마지막으로 tool이 실제로 DB를 업데이트했는지 확인합니다.


In [7]:
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
cur.execute('SELECT id, title, content FROM memos ORDER BY id DESC LIMIT 5')
rows = cur.fetchall()
conn.close()

print('최근 저장된 메모:')
for row in rows:
    print(row)

최근 저장된 메모:
(1, 'MCP Tool 실습', 'FastMCP로 첫 번째 tool을 만들고 SQLite에 저장했습니다.')


## 실습 정리

이 실습에서 확인한 핵심:

1. `@mcp.tool`로 일반 Python 함수를 MCP tool로 노출할 수 있다.
2. 함수 이름, docstring, 타입 힌트로 tool 이름/설명/입력 스키마가 자동 생성된다.
3. Client는 `list_tools()`로 tool과 schema를 확인할 수 있다.
4. Client는 `call_tool()`로 실제 tool을 실행할 수 있다.
5. tool은 계산뿐 아니라 DB 저장 같은 실제 action도 수행할 수 있다.
